# 01 · Develop the training job in SMUS JupyterLab

You are the data scientist. You are building a FashionMNIST classifier. You
want fast iteration: train on a small slice, look at the loss curve, tweak,
rerun. When you are happy with the model, run `launch_to_slurm.py` from a
JupyterSpace terminal to submit the same code to the on-prem GPU.

The training code lives in `on-prem/training/train.py` as a plain Python
module exposing `train(epochs, batch_size, lr, subset, ...)`. Same module
runs here and on the cluster — nothing is notebook-specific.


## Train on a 2 000-sample subset


In [ ]:
import sys
sys.path.insert(0, "../on-prem/training")

from train import train

result = train(epochs=2, subset=2000, run_name="notebook-dev")
result


## Inspect what MLflow captured

`train()` logs params, metrics, and an artifact-URI tag to whatever tracking
server `MLFLOW_TRACKING_URI` points at. If your SMUS kernel has it set to
the managed server, the run is already there. Otherwise MLflow defaults
to a local `./mlruns` directory next to this notebook.


In [ ]:
import mlflow

runs = mlflow.search_runs(experiment_names=["smus-to-dgx"], max_results=5)
runs[["run_id", "status", "metrics.val_acc", "metrics.train_loss", "params.epochs", "params.subset"]]


## Plot the loss curve


In [ ]:
import matplotlib.pyplot as plt

client = mlflow.tracking.MlflowClient()
train_loss = client.get_metric_history(result["run_id"], "train_loss")
val_loss   = client.get_metric_history(result["run_id"], "val_loss")

plt.figure(figsize=(6, 3))
plt.plot([m.step for m in train_loss], [m.value for m in train_loss], marker="o", label="train")
plt.plot([m.step for m in val_loss],   [m.value for m in val_loss],   marker="o", label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
plt.title(f"dev run · val_acc={result['val_acc']:.3f}");


---

Happy with the model? Edit the `CONFIG` block at the top of
`launch_to_slurm.py` and run `python launch_to_slurm.py` from a JupyterSpace
terminal to fire the same `train()` at full scale on the on-prem GB10.
